In [32]:
pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('data/AmesHousing.csv');

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   object 
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2930 non-null   object 
 9   Land Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot Config       2930 non-null   object 
 12  Land Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition 1      2930 non-null   object 
 15  Condition 2      2930 non-null   object 
 16  Bldg Type        2930 non-null   object 
 17  House Style   

In [5]:
data.describe(include='object')

,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
count,2930,2930,198,2930,2930,2930,2930,2930,2930,2930,...,2773,2771,2771,2771,2930,13,572,106,2930,2930
unique,7,2,2,4,4,3,5,3,28,9,...,6,3,5,5,3,4,4,5,10,6
top,RL,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,...,Attchd,Unf,TA,TA,Y,Ex,MnPrv,Shed,WD,Normal
freq,2273,2918,120,1859,2633,2927,2140,2789,443,2522,...,1731,1231,2615,2665,2652,4,330,95,2536,2413


In [6]:
a = pd.DataFrame()
a['columns'] = list(data.columns)
a['nulls'] = list(data.isna().sum())

In [7]:
pd.options.display.max_rows = 90

In [8]:
a

,columns,nulls
0,Order,0
1,PID,0
2,MS SubClass,0
3,MS Zoning,0
4,Lot Frontage,490
5,Lot Area,0
6,Street,0
7,Alley,2732
8,Lot Shape,0
9,Land Contour,0


In [16]:
for i in a[a['nulls']>500]['columns'].values:
    data.drop(i,axis=1,inplace=True,errors='ignore')

In [14]:
data.isna().sum() 

Order                0
PID                  0
MS SubClass          0
MS Zoning            0
Lot Frontage       490
Lot Area             0
Street               0
Lot Shape            0
Land Contour         0
Utilities            0
Lot Config           0
Land Slope           0
Neighborhood         0
Condition 1          0
Condition 2          0
Bldg Type            0
House Style          0
Overall Qual         0
Overall Cond         0
Year Built           0
Year Remod/Add       0
Roof Style           0
Roof Matl            0
Exterior 1st         0
Exterior 2nd         0
Mas Vnr Area        23
Exter Qual           0
Exter Cond           0
Foundation           0
Bsmt Qual           80
Bsmt Cond           80
Bsmt Exposure       83
BsmtFin Type 1      80
BsmtFin SF 1         1
BsmtFin Type 2      81
BsmtFin SF 2         1
Bsmt Unf SF          1
Total Bsmt SF        1
Heating              0
Heating QC           0
Central Air          0
Electrical           1
1st Flr SF           0
2nd Flr SF 

In [17]:
data.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Area', 'Exter Qual', 'Exter Cond',
       'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Garage Type',
       'Garage Yr Blt', 'Garage Finish', 'Garage Cars', 'Garage Area',


In [18]:
data['Price_Cat'] = pd.qcut(data['SalePrice'],q=3,labels=['low','medium','high'])

In [19]:
data

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,...,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Price_Cat
0,1,526301100,20,RL,141.0,31770,Pave,IR1,Lvl,AllPub,...,0,0,0,0,5,2010,WD,Normal,215000,high
1,2,526350040,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,...,0,120,0,0,6,2010,WD,Normal,105000,low
2,3,526351010,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,...,0,0,0,12500,6,2010,WD,Normal,172000,medium
3,4,526353030,20,RL,93.0,11160,Pave,Reg,Lvl,AllPub,...,0,0,0,0,4,2010,WD,Normal,244000,high
4,5,527105010,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,...,0,0,0,0,3,2010,WD,Normal,189900,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,IR1,Lvl,AllPub,...,0,0,0,0,3,2006,WD,Normal,142500,medium
2926,2927,923276100,20,RL,NaN,8885,Pave,IR1,Low,AllPub,...,0,0,0,0,6,2006,WD,Normal,131000,low
2927,2928,923400125,85,RL,62.0,10441,Pave,Reg,Lvl,AllPub,...,0,0,0,700,7,2006,WD,Normal,132000,low
2928,2929,924100070,20,RL,77.0,10010,Pave,Reg,Lvl,AllPub,...,0,0,0,0,4,2006,WD,Normal,170000,medium


In [20]:
data.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Area', 'Exter Qual', 'Exter Cond',
       'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Garage Type',
       'Garage Yr Blt', 'Garage Finish', 'Garage Cars', 'Garage Area',


In [21]:
X = data.drop(columns=['Order','PID','SalePrice','Price_Cat'])
y_reg = data['SalePrice']
y_cls = data['Price_Cat']

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
X_train_reg,X_test_reg,y_train_reg,y_test_reg = train_test_split(X,y_reg,test_size=0.2)

In [24]:
X_train_cls,X_test_cls,y_train_cls,y_test_cls = train_test_split(X,y_cls,test_size=0.2)

In [25]:
X_train_cls.shape

(2344, 73)

In [27]:
X_train_reg.shape

(2344, 73)

In [28]:
y_train_cls.replace({'low':0,'medium':1,'high':2},inplace=True)
y_test_cls.replace({'low':0,'medium':1,'high':2},inplace=True) 

In [30]:
y_test_cls

103     1
690     0
1665    1
1671    0
1340    0
       ..
2427    2
2877    0
1423    1
2315    1
855     1
Name: Price_Cat, Length: 586, dtype: category
Categories (3, int64): [0 < 1 < 2]

In [34]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.ensemble import AdaBoostClassifier,AdaBoostRegressor,GradientBoostingClassifier,GradientBoostingRegressor
from xgboost import XGBClassifier,XGBRegressor
from sklearn.metrics import accuracy_score,mean_squared_error,r2_score
from sklearn.compose import ColumnTransformer

In [41]:
num = X.select_dtypes(include=['int64','float64']).columns
cat = X.select_dtypes(include=['object']).columns

In [42]:
cat

Index(['MS Zoning', 'Street', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Exter Qual', 'Exter Cond',
       'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
       'Paved Drive', 'Sale Type', 'Sale Condition'],
      dtype='object')

In [45]:
num_pipe = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_pipe = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])
prep_pipe = ColumnTransformer(transformers=[
    ('num',num_pipe,num),
    ('cat',cat_pipe,cat) 
])
reg = {
    'Adaboost':AdaBoostRegressor(random_state=42),
    'GradientBoosting':GradientBoostingRegressor(random_state=42),
    'XGBoost':XGBRegressor(random_state=42)
}
cls = {
    'Adaboost':AdaBoostClassifier(),
    'GradientBoosting':GradientBoostingClassifier(),
    'XGBoost':XGBClassifier()
}

In [47]:
import numpy as np

In [48]:
for name,model in reg.items():
    print('======================================')
    pipe = Pipeline(steps=[
        ('preprocess',prep_pipe),
        ('model',model)
    ])
    pipe.fit(X_train_reg,y_train_reg)
    print('Fitting for',name,'Regressor')
    y_pred = pipe.predict(X_test_reg)
    rmse = np.sqrt(mean_squared_error(y_test_reg,y_pred))
    r2 = r2_score(y_test_reg,y_pred)
    print(f'{name}_rmse = {rmse}')
    print(f'{name}_r2 = {r2}')
    print('____________________________________\n\n')
    

Fitting for Adaboost Regressor
Adaboost_rmse = 33366.64691493275
Adaboost_r2 = 0.7967461540580087
____________________________________


Fitting for GradientBoosting Regressor
GradientBoosting_rmse = 20295.964928881305
GradientBoosting_r2 = 0.9247973666936469
____________________________________


Fitting for XGBoost Regressor
XGBoost_rmse = 24429.27817189857
XGBoost_r2 = 0.8910480737686157
____________________________________




In [49]:
for name,model in cls.items():
    print('======================================')
    pipe = Pipeline(steps=[
        ('preprocess',prep_pipe),
        ('model',model)
    ])
    pipe.fit(X_train_cls,y_train_cls)
    print('Fitting for',name,'Classifier')
    y_pred = pipe.predict(X_test_cls)
    rmse = np.sqrt(mean_squared_error(y_test_reg,y_pred))
    acc = accuracy_score(y_test_cls,y_pred)
    print(f'{name}_accuracy = {acc}')
    print('____________________________________\n\n')    

Fitting for Adaboost Classifier
Adaboost_accuracy = 0.7935153583617748
____________________________________


Fitting for GradientBoosting Classifier
GradientBoosting_accuracy = 0.841296928327645
____________________________________


Fitting for XGBoost Classifier
XGBoost_accuracy = 0.8430034129692833
____________________________________


